# NLP Disaster Tweets — Ensemble v2: DistilBERT + BERTweet + Self-Cleaning

Следующая итерация поверх `ensemble.ipynb` (public LB 0.83787). Добавляем data-driven label cleaning.

**Идея:** в датасете ~7600 твитов есть шум в разметке. Часть твитов размечена неправильно:
- **Конфликты** — один и тот же текст помечен и 0, и 1 (≈18 пар в train)
- **Уверенно неправильные** — обе модели в ensemble OOF дают вероятность > 0.9, а метка = 0 (или наоборот)

**Подход:**
1. Разрешаем конфликты мажоритарным голосом
2. Загружаем `distil_oof.npy` и `bert_oof.npy` из прошлого прогона `ensemble.ipynb`
3. Находим examples где `ensemble_oof > 0.90` но `y = 0` (или `< 0.10` но `y = 1`) — флипаем их метку
4. Переобучаем ensemble на cleaned train
5. Тот же финал: threshold tuning по OOF → overlap fix → submission

**Почему это работает (а не circular reasoning):**  
Когда обе модели независимо ошибаются в одну сторону на OOF (не видели этот пример в обучении), вероятнее всего ошибка не в моделях, а в разметке. High-threshold (0.90/0.10) даёт ~50-100 флипов из 7613 — точечная чистка.

**Требование:** должны существовать `distil_oof.npy` и `bert_oof.npy` от прошлого `ensemble.ipynb`.

## 1. Импорты и настройки

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps');  NUM_GPUS = 1
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda'); NUM_GPUS = torch.cuda.device_count()
else:
    DEVICE = torch.device('cpu');  NUM_GPUS = 0

NUM_WORKERS = 4 if DEVICE.type == 'cuda' else 0
PIN_MEMORY  = DEVICE.type == 'cuda'

print(f'Device: {DEVICE} | GPUs: {NUM_GPUS} | PyTorch: {torch.__version__}')

## 2. Загрузка данных

In [ ]:
df_train = pd.read_csv('data/train.csv')
df_test  = pd.read_csv('data/test.csv')
print(f'Train: {df_train.shape},  Test: {df_test.shape}')

## 3. Загрузка OOF probs от прошлого ensemble

Нужны `distil_oof.npy` и `bert_oof.npy` от `ensemble.ipynb`. Если их нет — запустить тот ноутбук сначала.

In [ ]:
assert os.path.exists('distil_oof.npy') and os.path.exists('bert_oof.npy'), \
    'Нужны OOF probs от ensemble.ipynb. Запусти его сначала.'

distil_oof_old = np.load('distil_oof.npy')
bert_oof_old   = np.load('bert_oof.npy')
ensemble_oof_old = (distil_oof_old + bert_oof_old) / 2

assert len(distil_oof_old) == len(df_train), f'OOF size mismatch: {len(distil_oof_old)} vs {len(df_train)}'

y_orig = df_train['target'].values
print(f'Loaded OOFs for {len(ensemble_oof_old)} train rows')
print(f'Old ensemble OOF F1@0.5: {f1_score(y_orig, (ensemble_oof_old > 0.5).astype(int)):.4f}')

## 4. Conflict resolution — дедупликация train по тексту

Один текст с разными метками → берём мажоритарный, при ничьей → 1. Чистые дубликаты схлопываются.

In [ ]:
def majority_label(labels):
    counts = labels.value_counts()
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        return 1
    return counts.idxmax()

label_counts = df_train.groupby('text')['target'].nunique()
n_conflicts = (label_counts > 1).sum()
print(f'Конфликтующих текстов в train: {n_conflicts}')

resolved = df_train.groupby('text', sort=False)['target'].agg(majority_label).reset_index()
first_meta = df_train.groupby('text', sort=False)[['keyword', 'location']].first().reset_index()
df_train_dedup = resolved.merge(first_meta, on='text')

print(f'Train: {len(df_train)} → {len(df_train_dedup)} строк после дедупликации')

first_pos = df_train.reset_index().groupby('text', sort=False)['index'].first()
dedup_oof = ensemble_oof_old[first_pos.loc[df_train_dedup['text']].values]
print(f'OOF aligned to dedup train: shape={dedup_oof.shape}')

## 5. High-confidence label flips

Флипаем там, где обе модели согласно ошибаются на OOF с большим отрывом.

In [ ]:
FLIP_HIGH = 0.90
FLIP_LOW  = 0.10

y_dedup = df_train_dedup['target'].values.copy()

flip_to_1 = (dedup_oof > FLIP_HIGH) & (y_dedup == 0)
flip_to_0 = (dedup_oof < FLIP_LOW)  & (y_dedup == 1)

print(f'Flip 0 → 1 (модели уверенно говорят "disaster"):  {flip_to_1.sum()} примеров')
print(f'Flip 1 → 0 (модели уверенно говорят "not disaster"): {flip_to_0.sum()} примеров')
print(f'Всего флипов: {flip_to_1.sum() + flip_to_0.sum()} из {len(y_dedup)}')

print('\nПримеры флипов 0 → 1 (модели считают катастрофой, размечено как не-катастрофа):')
examples = df_train_dedup[flip_to_1].head(5)
for txt, prob in zip(examples['text'], dedup_oof[flip_to_1][:5]):
    print(f'  [prob={prob:.3f}] {txt[:120]}')

print('\nПримеры флипов 1 → 0 (модели считают не-катастрофой, размечено как катастрофа):')
examples = df_train_dedup[flip_to_0].head(5)
for txt, prob in zip(examples['text'], dedup_oof[flip_to_0][:5]):
    print(f'  [prob={prob:.3f}] {txt[:120]}')

y_dedup[flip_to_1] = 1
y_dedup[flip_to_0] = 0
df_train_clean = df_train_dedup.copy()
df_train_clean['target'] = y_dedup

print(f'\nDistribution до: {dict(zip(*np.unique(df_train_dedup["target"], return_counts=True)))}')
print(f'Distribution после: {dict(zip(*np.unique(y_dedup, return_counts=True)))}')

## 6. Препроцессинг под обе модели

Как в `ensemble.ipynb`: агрессивная очистка для DistilBERT (только text), мягкая нормализация + keyword/location для BERTweet.

In [ ]:
def clean_for_distilbert(text: str) -> str:
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'[^\w\s.,!?\'-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_for_bertweet(text: str) -> str:
    text = re.sub(r'http\S+|www\S+', 'HTTPURL', text)
    text = re.sub(r'@\w+', '@USER', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_bertweet_input(row) -> str:
    keyword  = str(row['keyword']).replace('%20', ' ') if pd.notna(row['keyword'])  else ''
    location = str(row['location'])                    if pd.notna(row['location']) else ''
    text     = row['_bert_clean']
    parts = [p for p in [keyword, location, text] if p]
    return ' | '.join(parts)

df_train_clean['text_distilbert'] = df_train_clean['text'].apply(clean_for_distilbert)
df_test['text_distilbert']        = df_test['text'].apply(clean_for_distilbert)

df_train_clean['_bert_clean'] = df_train_clean['text'].apply(normalize_for_bertweet)
df_test['_bert_clean']        = df_test['text'].apply(normalize_for_bertweet)
df_train_clean['text_bertweet'] = df_train_clean.apply(build_bertweet_input, axis=1)
df_test['text_bertweet']        = df_test.apply(build_bertweet_input, axis=1)

print(f'Cleaned train: {len(df_train_clean)} строк')

## 7. Универсальная KFold-функция

In [ ]:
MAX_LEN       = 128
EPOCHS        = 3
LEARNING_RATE = 2e-5
N_SPLITS      = 5
BATCH_SIZE    = 32 * max(NUM_GPUS, 1)


class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, labels=None):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=MAX_LEN, return_tensors='pt'
        )
        self.labels = labels.reset_index(drop=True) if labels is not None else None

    def __len__(self):
        return self.encodings['input_ids'].shape[0]

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return item


def predict_probs(model, loader, has_labels=True):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            probs.extend(torch.softmax(out.logits, dim=1)[:, 1].cpu().numpy())
            if has_labels:
                labels.extend(batch['labels'].cpu().numpy())
    return np.array(probs), (np.array(labels) if has_labels else None)


def train_kfold(model_name, train_texts, test_texts, y, tokenizer, tag=''):
    test_ds = TextDataset(pd.Series(test_texts), tokenizer)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    oof_probs  = np.zeros(len(train_texts))
    test_probs = np.zeros(len(test_texts))
    fold_scores = []

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_texts, y), 1):
        print(f'\n--- [{tag}] Fold {fold}/{N_SPLITS} ---')

        tr_ds = TextDataset(pd.Series(train_texts[tr_idx]), tokenizer, pd.Series(y[tr_idx]))
        va_ds = TextDataset(pd.Series(train_texts[va_idx]), tokenizer, pd.Series(y[va_idx]))

        tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE,
                               num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
        model = model.to(DEVICE)
        if NUM_GPUS > 1:
            model = nn.DataParallel(model)
        base = model.module if isinstance(model, nn.DataParallel) else model

        optimizer = torch.optim.AdamW(base.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
        total_steps = EPOCHS * len(tr_loader)
        scheduler = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)

        val_p, val_y = None, None
        for ep in range(1, EPOCHS + 1):
            model.train()
            for batch in tqdm(tr_loader, desc=f'  train ep{ep}', leave=False):
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                out = model(**batch)
                loss = out.loss.mean()
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(base.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            val_p, val_y = predict_probs(model, va_loader)
            print(f'  ep{ep} val F1@0.5: {f1_score(val_y, (val_p > 0.5).astype(int)):.4f}')

        oof_probs[va_idx] = val_p
        fold_scores.append(f1_score(val_y, (val_p > 0.5).astype(int)))

        test_p, _ = predict_probs(model, test_loader, has_labels=False)
        test_probs += test_p / N_SPLITS

        del model, base, optimizer, scheduler
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    print(f'\n[{tag}] Mean fold F1@0.5: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
    print(f'[{tag}] Overall OOF F1@0.5: {f1_score(y, (oof_probs > 0.5).astype(int)):.4f}')
    return oof_probs, test_probs, fold_scores

## 8. DistilBERT на cleaned train

In [ ]:
y_clean = df_train_clean['target'].values

distil_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

distil_oof, distil_test, distil_scores = train_kfold(
    'distilbert-base-uncased',
    df_train_clean['text_distilbert'].values,
    df_test['text_distilbert'].values,
    y_clean,
    distil_tokenizer,
    tag='DistilBERT'
)

np.save('distil_oof_v2.npy',  distil_oof)
np.save('distil_test_v2.npy', distil_test)

## 9. BERTweet на cleaned train

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained('vinai/bertweet-base', use_fast=False, normalization=False)

bert_oof, bert_test, bert_scores = train_kfold(
    'vinai/bertweet-base',
    df_train_clean['text_bertweet'].values,
    df_test['text_bertweet'].values,
    y_clean,
    bert_tokenizer,
    tag='BERTweet'
)

np.save('bert_oof_v2.npy',  bert_oof)
np.save('bert_test_v2.npy', bert_test)

## 10. Ensemble + threshold tuning

In [ ]:
oof_ensemble  = (distil_oof  + bert_oof)  / 2
test_ensemble = (distil_test + bert_test) / 2

print('--- OOF F1@0.5 (на cleaned train) ---')
print(f'DistilBERT : {f1_score(y_clean, (distil_oof   > 0.5).astype(int)):.4f}')
print(f'BERTweet   : {f1_score(y_clean, (bert_oof     > 0.5).astype(int)):.4f}')
print(f'Ensemble   : {f1_score(y_clean, (oof_ensemble > 0.5).astype(int)):.4f}')

thresholds = np.arange(0.30, 0.61, 0.01)
f1_per_t = [f1_score(y_clean, (oof_ensemble > t).astype(int)) for t in thresholds]
best_idx = int(np.argmax(f1_per_t))
best_threshold = thresholds[best_idx]
best_oof_f1 = f1_per_t[best_idx]

print(f'\nBest ensemble threshold: {best_threshold:.2f}  →  OOF F1 = {best_oof_f1:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_per_t, marker='o', markersize=3)
ax.axvline(best_threshold, color='red', linestyle='--', label=f'best={best_threshold:.2f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('OOF F1 (ensemble, cleaned)')
ax.set_title('Ensemble v2: F1 vs threshold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Classification report + confusion matrix

In [ ]:
oof_preds = (oof_ensemble > best_threshold).astype(int)
print(classification_report(y_clean, oof_preds, target_names=['Not Disaster', 'Disaster']))

cm = confusion_matrix(y_clean, oof_preds)
ConfusionMatrixDisplay(cm, display_labels=['Not Disaster', 'Disaster']).plot(cmap='Blues')
plt.title(f'Ensemble v2 OOF CM (F1={best_oof_f1:.4f}, t={best_threshold:.2f})')
plt.show()

## 12. Submission с overlap fix

In [ ]:
test_preds = (test_ensemble > best_threshold).astype(int).tolist()
print(f'До overlap fix → Disaster: {sum(test_preds)}, Not disaster: {len(test_preds) - sum(test_preds)}')

overlap_labels = (
    df_train_clean[df_train_clean['text'].isin(df_test['text'])]
    .set_index('text')['target']
    .to_dict()
)

overridden = 0
for i, text in enumerate(df_test['text']):
    if text in overlap_labels:
        test_preds[i] = int(overlap_labels[text])
        overridden += 1

print(f'Overlap fix: перезаписано {overridden} предсказаний ({len(overlap_labels)} уникальных текстов)')
print(f'После overlap fix → Disaster: {sum(test_preds)}, Not disaster: {len(test_preds) - sum(test_preds)}')

submission = pd.read_csv('data/sample_submission.csv')
submission['target'] = test_preds
submission.to_csv('submission_ensemble_v2.csv', index=False)

print('\nsubmission_ensemble_v2.csv saved')
submission.head()